# Chapter 8 — The Transformer Block

Every number in Chapter 8, recomputed with `tfs/model.py` (`RMSNorm`, `SwiGLU`, `Block`,
`Config.n_params`). Each cell **asserts** what it checks, so `python tests/run_notebooks.py 08`
fails if the chapter and the code disagree. Exercise 8.4 needs GPT-2's weights and is not
reproduced.

In [ ]:
import math, sys, os
sys.path.insert(0, os.path.abspath(".."))      # run from notebooks/
sys.path.insert(0, os.path.abspath("."))       # or from the repo root
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(0)
from tfs.model import Config, Model, RMSNorm, SwiGLU

## 8.2 — the worked example: LayerNorm and RMSNorm on x = [3, 1, 4, 8]

**DISCREPANCY (typo).** The chapter's code-cell comment prints the RMSNorm output as
`[ 0.633, 0.211, 0.843, 1.687]`; the prose above it (correctly) says `0.632`.
3 / √22.5 = 0.63246, which rounds to 0.632 with or without the 1e-5 epsilon.

In [ ]:
x = torch.tensor([3., 1., 4., 8.])
mu, var = x.mean(), x.var(unbiased=False)
assert mu.item() == 4 and var.item() == 6.5 and round(var.sqrt().item(), 4) == 2.5495
ln = (x - mu) / torch.sqrt(var + 1e-5)
ms = x.pow(2).mean()
assert ms.item() == 22.5 and round(ms.sqrt().item(), 4) == 4.7434
rms = x * torch.rsqrt(ms + 1e-5)
r3 = lambda t: [round(v, 3) for v in t.tolist()]
print("LN ", r3(ln), "  RMS", r3(rms))
assert r3(ln) == [-0.392, -1.177, 0.0, 1.569]
assert r3(rms) == [0.632, 0.211, 0.843, 1.687]                               # not 0.633
assert abs(ln.mean()) < 1e-6 and abs(ln.std(unbiased=False) - 1) < 1e-4
assert round(rms.mean().item(), 3) == 0.843 and abs(rms.pow(2).mean().sqrt() - 1) < 1e-4
cos = F.cosine_similarity(ln, rms, dim=0).item()
print(f"cosine(LN, RMS) = {cos:.4f}")
assert round(cos, 2) == 0.54
# the module in tfs/model.py gives the same RMSNorm
assert torch.allclose(RMSNorm(4, eps=1e-5)(x), rms, atol=1e-6)
# shift every element: LayerNorm is unchanged, RMSNorm moves
ln2 = (x + 10 - (x + 10).mean()) / torch.sqrt((x + 10).var(unbiased=False) + 1e-5)
assert torch.allclose(ln, ln2, atol=1e-5) and not torch.allclose(RMSNorm(4)(x + 10), RMSNorm(4)(x), atol=1e-2)

## 8.2 — how close LN and RMSNorm are on realistic vectors (d = 768)

The chapter: median cosine 0.9997 over 2000 random zero-mean vectors, worst case 0.994, and 0.45
once the mean is pushed 2σ off zero (1/√5 = 0.447 analytically).

In [ ]:
torch.manual_seed(0)
def cos_ln_rms(v):
    l = F.layer_norm(v, (v.shape[-1],), eps=1e-5)
    r = v * torch.rsqrt(v.pow(2).mean(-1, keepdim=True) + 1e-5)
    return F.cosine_similarity(l, r, dim=-1)
c = cos_ln_rms(torch.randn(2000, 768))
c_shift = cos_ln_rms(torch.randn(2000, 768) + 2.0)
print(f"median {c.median():.4f}  worst {c.min():.4f}  shifted by 2 sigma: {c_shift.median():.3f}")
assert round(c.median().item(), 4) == 0.9997 and 0.99 < c.min().item() < 0.999
assert round(c_shift.median().item(), 2) == 0.45 and abs(c_shift.median().item() - 1 / math.sqrt(5)) < 0.01

## 8.2 — LayerNorm ≠ BatchNorm: a token's output never depends on the rest of the batch

In [ ]:
torch.manual_seed(0)
a, others = torch.randn(1, 5, 32), torch.randn(7, 5, 32)
lnm = nn.LayerNorm(32)
assert torch.allclose(lnm(a), lnm(torch.cat([a, others]))[:1], atol=1e-6)
bn = nn.BatchNorm1d(32).train()
alone = bn(a.reshape(-1, 32))
assert not torch.allclose(alone, bn(torch.cat([a, others]).reshape(-1, 32))[:5], atol=1e-3)

## 8.2 — pre-LN never normalises the stream, so its magnitude grows with depth (hence a final norm)

In [ ]:
torch.manual_seed(0)
m = Model(Config(vocab_size=256, d_model=64, n_layer=8, n_head=4, n_kv_head=2, d_head=16, d_ff=128, max_T=64))
rms_by_layer = []
hooks = [b.register_forward_hook(lambda mod, i, o: rms_by_layer.append(o[0].pow(2).mean().sqrt().item()))
         for b in m.blocks]
m(torch.randint(0, 256, (4, 64)))
for h in hooks: h.remove()
print("stream RMS after each block:", [f"{v:.5f}" for v in rms_by_layer])
assert rms_by_layer == sorted(rms_by_layer) and rms_by_layer[-1] > rms_by_layer[0]
assert isinstance(m.norm, RMSNorm)                     # the final norm before the LM head

## 8.3 — the MLP is a per-position function: changing other tokens cannot change its output

In [ ]:
cfg = Config(vocab_size=256, d_model=64, n_layer=1, n_head=4, n_kv_head=2, d_head=16, d_ff=172, max_T=64)
mlp = SwiGLU(cfg)
h = torch.randn(1, 10, 64)
h2 = h.clone(); h2[0, 3:] = torch.randn(7, 64)
assert torch.allclose(mlp(h)[0, :3], mlp(h2)[0, :3])

## 8.3 — two-thirds of the model is in the MLP: 12C² per layer, GPT-2 small, Llama-2-7B

In [ ]:
C = 768
attn, mlp4 = 4 * C * C, 8 * C * C
assert mlp4 == 2 * attn and attn + mlp4 == 12 * C * C
body = 12 * 12 * C ** 2
print(f"GPT-2 small: 12*12*768^2 = {body / 1e6:.1f}M + 38.6M embedding = {(body + 50257 * 768) / 1e6:.1f}M")
assert round(body / 1e6) == 85 and round((body + 50257 * 768) / 1e6) == 124
llama = Config(vocab_size=32000, d_model=4096, n_layer=32, n_head=32, n_kv_head=32, d_head=128,
               d_ff=11008, tie_embed=False)
per = 3 * 4096 * 11008
print(f"Llama-2-7B: {llama.n_params():,} params; MLP {per / 1e6:.2f}M/layer x32 = {32 * per / 1e9:.2f}B"
      f" = {32 * per / llama.n_params():.1%} (of 6.7B: {32 * per / 6.7e9:.1%})")
assert llama.n_params() == 6_738_415_616
assert round(per / 1e6, 2) == 135.27 and round(4096 * 11008 / 1e6, 2) == 45.09
assert round(32 * per / 1e9, 2) == 4.33 and round(100 * 32 * per / 6.7e9) == 65
assert 11008 / 4096 == 2.6875 and 11008 % 256 == 0

## Exercise 8.3 — `swiglu_ff_dim`, and Llama-2-70B's 28672

In [ ]:
def swiglu_ff_dim(d_model, multiple_of=256, ffn_dim_multiplier=None):
    hidden = int(2 * 4 * d_model / 3)
    if ffn_dim_multiplier is not None:
        hidden = int(ffn_dim_multiplier * hidden)
    return multiple_of * ((hidden + multiple_of - 1) // multiple_of)
print(swiglu_ff_dim(4096), swiglu_ff_dim(8192), swiglu_ff_dim(8192, 4096, 1.3))
assert swiglu_ff_dim(4096) == 11008 and swiglu_ff_dim(8192) == 22016
assert swiglu_ff_dim(8192, multiple_of=4096, ffn_dim_multiplier=1.3) == 28672   # Llama-2-70B's config
# parameter match: SwiGLU at (8/3)C has the parameters of a GELU MLP at 4C
assert abs(3 * 4096 * swiglu_ff_dim(4096) / (2 * 4096 * 4 * 4096) - 1) < 0.01

## Exercise 8.1 — both norms on a random (4, 128, 512) input, verified and timed

The chapter: cosine typically > 0.98; RMSNorm "usually 20–40% faster in a pure-Python benchmark".
The speed ratio depends heavily on the machine and on how `var` is implemented; on this CPU the
hand-written RMSNorm is several times faster than the hand-written LayerNorm, while PyTorch's
fused `nn.LayerNorm` beats both. The assert is only that the hand-written RMSNorm is faster.

In [ ]:
import time
class LayerNorm(nn.Module):
    def __init__(self, d, eps=1e-5):
        super().__init__()
        self.g = nn.Parameter(torch.ones(d)); self.b = nn.Parameter(torch.zeros(d)); self.eps = eps
    def forward(self, x):
        mu = x.mean(-1, keepdim=True)
        var = x.var(-1, keepdim=True, unbiased=False)
        return (x - mu) / torch.sqrt(var + self.eps) * self.g + self.b
torch.manual_seed(0)
x = torch.randn(4, 128, 512)
mine, ref, rn = LayerNorm(512), nn.LayerNorm(512), RMSNorm(512)
with torch.no_grad():
    assert torch.allclose(mine(x), ref(x), atol=1e-5)
    lo, ro = mine(x), rn(x)
    assert lo.mean(-1).abs().max() < 1e-5 and (lo.std(-1, unbiased=False) - 1).abs().max() < 1e-3
    assert (ro.pow(2).mean(-1).sqrt() - 1).abs().max() < 1e-3
    cs = F.cosine_similarity(lo, ro, dim=-1)
    print(f"cosine LN vs RMS: min {cs.min():.4f}")
    assert cs.min() > 0.98
    def bench(f, reps=200):
        for _ in range(5): f(x)
        t = time.perf_counter()
        for _ in range(reps): f(x)
        return (time.perf_counter() - t) / reps
    t_ln, t_rms = bench(mine), bench(lambda v: v * torch.rsqrt(v.pow(2).mean(-1, keepdim=True) + 1e-6))
print(f"hand-written LN {t_ln * 1e3:.2f} ms   hand-written RMS {t_rms * 1e3:.2f} ms   ({t_ln / t_rms:.1f}x)")
assert t_rms < t_ln

## Exercise 8.2 — pre-LN vs post-LN gradients across 24 layers

Built with PyTorch's standard `nn.TransformerEncoderLayer` (`norm_first=False` is post-LN,
`True` is pre-LN; LayerNorm, default init), one forward + backward on random data, gradient norm
of each layer's attention input projection.

**DISCREPANCY (magnitude).** The direction matches the chapter: post-LN gradients are largest at
the top and smaller at layer 0, pre-LN gradients are roughly flat. But the size of the effect is
~5–10×, not "2–4 orders of magnitude". (With this course's own init — weights at std 0.02 and
un-normalised embeddings — a post-LN variant does show a ~10³ ratio, but only because layer 0
alone sees the tiny raw embedding; layers 1–23 are flat. That is an init artefact, not geometric
decay.) The well-known result (Xiong et al. 2020) is the direction; the "2–4 orders" figure
should be softened or re-measured.

In [ ]:
def grad_profile(norm_first, L=24, d=64):
    torch.manual_seed(0)
    layer = nn.TransformerEncoderLayer(d, 4, 4 * d, dropout=0.0, batch_first=True, norm_first=norm_first)
    enc = nn.TransformerEncoder(layer, L, enable_nested_tensor=False)
    out = enc(torch.randn(8, 32, d))
    if norm_first:
        out = F.layer_norm(out, (d,))                 # pre-LN needs its final norm
    (out @ torch.randn(d)).pow(2).mean().backward()
    return [l.self_attn.in_proj_weight.grad.norm().item() for l in enc.layers]
post, pre = grad_profile(False), grad_profile(True)
print("post-LN", " ".join(f"{v:.1e}" for v in post[::4]), f"  top/bottom {post[-1] / post[0]:.1f}x")
print("pre-LN ", " ".join(f"{v:.1e}" for v in pre[::4]), f"  top/bottom {pre[-1] / pre[0]:.1f}x")
assert post[-1] / post[0] > 2                          # post-LN: smaller at layer 0 ...
assert post[-1] / post[0] < 100                        # ... but not by 2-4 orders of magnitude
assert 0.3 < pre[-1] / pre[0] < 3                      # pre-LN: roughly flat